# audio

> Wav2vec 2.0 encoder, as `Wav2Vec2Model` runs it at inference.
> 
> Dropout, SpecAugment and LayerDrop are the identity. This is the base
> wiring: group norm in the first convolution only, and post-LayerNorm
> transformer blocks.


In [ ]:
#| default_exp audio


In [ ]:
#| hide
from nbdev.showdoc import *


In [ ]:
#| export
from __future__ import annotations


In [ ]:
#| export
from max.experimental import functional as F
from max.experimental.nn import LayerNorm, Linear, Module, ModuleList
from max.experimental.tensor import Tensor

from nada.config import Wav2Vec2Config
from nada.layers import ChannelNorm, Conv1d, Wav2Vec2Attention, Wav2Vec2FeedForward


## Feature encoder

Seven strided convolutions turn the raw waveform into 512-channel frames. Only the first has a norm.


In [ ]:
#| export
class Wav2Vec2ConvLayer(Module):
    """Convolution, GELU, and for the first layer a per-channel norm in between."""

    def __init__(self, config: Wav2Vec2Config, index: int) -> None:
        in_channels = 1 if index == 0 else config.conv_dim[index - 1]
        self.normalize = index == 0
        self.conv = Conv1d(
            in_channels,
            config.conv_dim[index],
            config.conv_kernel[index],
            stride=config.conv_stride[index],
            bias=config.conv_bias,
        )
        if self.normalize:
            self.layer_norm = ChannelNorm(config.conv_dim[index], config.group_norm_eps)

    def forward(self, hidden: Tensor) -> Tensor:
        hidden = self.conv(hidden)
        if self.normalize:
            hidden = self.layer_norm(hidden)
        return F.gelu(hidden)


In [ ]:
#| export
class Wav2Vec2FeatureEncoder(Module):
    """`(batch, samples)` in, `(batch, frames, conv_dim[-1])` out."""

    def __init__(self, config: Wav2Vec2Config) -> None:
        self.conv_layers = ModuleList([Wav2Vec2ConvLayer(config, i) for i in range(len(config.conv_dim))])

    def forward(self, input_values: Tensor) -> Tensor:
        batch, samples = input_values.shape[0], input_values.shape[1]
        hidden = input_values.reshape([batch, samples, 1])
        for layer in self.conv_layers:
            hidden = layer(hidden)
        return hidden


In [ ]:
#| export
class Wav2Vec2FeatureProjection(Module):
    """LayerNorm, then a linear map up to the transformer width."""

    def __init__(self, config: Wav2Vec2Config) -> None:
        self.layer_norm = LayerNorm(config.feature_size, eps=config.layer_norm_eps)
        self.projection = Linear(config.feature_size, config.hidden_size)

    def forward(self, features: Tensor) -> Tensor:
        return self.projection(self.layer_norm(features))


## Transformer

Positions come from a grouped convolution over the whole sequence, not from a table. Its kernel is 128 wide, so it is padded by 64 on each side. An even kernel makes the output one frame too long, and that last frame is dropped.


In [ ]:
#| export
class Wav2Vec2PositionalConvEmbedding(Module):
    """Grouped convolution, trim, GELU. The weight is the folded weight-norm weight."""

    def __init__(self, config: Wav2Vec2Config) -> None:
        kernel = config.num_conv_pos_embeddings
        self.trim = kernel % 2 == 0
        self.conv = Conv1d(
            config.hidden_size,
            config.hidden_size,
            kernel,
            padding=kernel // 2,
            groups=config.num_conv_pos_embedding_groups,
            bias=True,
        )

    def forward(self, hidden: Tensor) -> Tensor:
        steps = int(hidden.shape[1])
        out = self.conv(hidden)
        if self.trim:
            out = F.slice_tensor(out, [slice(None), slice(0, steps), slice(None)])
        return F.gelu(out)


In [ ]:
#| export
class Wav2Vec2EncoderLayer(Module):
    """Post-LN block: attention, add, norm, feed-forward, add, norm."""

    def __init__(self, config: Wav2Vec2Config) -> None:
        self.attention = Wav2Vec2Attention(config.hidden_size, config.num_attention_heads)
        self.layer_norm = LayerNorm(config.hidden_size, eps=config.layer_norm_eps)
        self.feed_forward = Wav2Vec2FeedForward(config.hidden_size, config.intermediate_size)
        self.final_layer_norm = LayerNorm(config.hidden_size, eps=config.layer_norm_eps)

    def forward(self, hidden: Tensor) -> Tensor:
        hidden = self.layer_norm(hidden + self.attention(hidden))
        return self.final_layer_norm(hidden + self.feed_forward(hidden))


In [ ]:
#| export
class Wav2Vec2Encoder(Module):
    """Add position embeddings, normalize, then run the transformer blocks."""

    def __init__(self, config: Wav2Vec2Config) -> None:
        self.pos_conv_embed = Wav2Vec2PositionalConvEmbedding(config)
        self.layer_norm = LayerNorm(config.hidden_size, eps=config.layer_norm_eps)
        self.layers = ModuleList([Wav2Vec2EncoderLayer(config) for _ in range(config.num_hidden_layers)])

    def forward(self, hidden: Tensor) -> Tensor:
        hidden = self.layer_norm(hidden + self.pos_conv_embed(hidden))
        for layer in self.layers:
            hidden = layer(hidden)
        return hidden


In [ ]:
#| export
class Wav2Vec2Model(Module):
    """Waveform in, one contextual `hidden_size` vector per 20 ms frame out."""

    def __init__(self, config: Wav2Vec2Config) -> None:
        self.feature_extractor = Wav2Vec2FeatureEncoder(config)
        self.feature_projection = Wav2Vec2FeatureProjection(config)
        self.encoder = Wav2Vec2Encoder(config)

    def forward(self, input_values: Tensor) -> Tensor:
        features = self.feature_extractor(input_values)
        return self.encoder(self.feature_projection(features))
